# Horse Racing RL — Training Notebook

Train jockey AI models using curriculum learning (SB3).

**Colab Pro tips:**
- Runtime → Change runtime type → **A100 GPU** or **V100** for fastest training
- The GPU accelerates PPO gradient updates; more parallel envs speed up rollout collection
- With Colab Pro's 51GB RAM, we run **16 parallel envs** with `SubprocVecEnv` for true multiprocessing

**Outputs:** ONNX model file for browser inference.

## 1. Setup

In [ ]:
import os

# Clone or pull latest
if os.path.exists("hr-simulation"):
    !cd hr-simulation && git pull
else:
    !git clone https://github.com/ue-too/hr-simulation.git

%cd hr-simulation

In [ ]:
# Install dependencies
!pip install -q stable-baselines3 gymnasium pettingzoo numpy torch onnx onnxruntime

In [ ]:
# Verify import
from horse_racing.engine import HorseRacingEngine
from horse_racing.types import HorseAction, HORSE_COUNT
from horse_racing.env import HorseRacingSingleEnv
from horse_racing.reward import compute_reward
print(f"Engine OK — {HORSE_COUNT} horses")

# Quick sanity check
engine = HorseRacingEngine("tracks/simple_oval.json")
engine.step([HorseAction() for _ in range(HORSE_COUNT)])
obs = engine.get_observations()
arr = engine.obs_to_array(obs[0])
print(f"Observation vector: {arr.shape} — {arr.dtype}")

## 2. Training Configuration

In [ ]:
import torch

# Auto-detect GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    N_ENVS = 16        # Colab Pro has enough RAM for 16
    BATCH_SIZE = 512    # larger batches to keep GPU busy
else:
    print("No GPU — using CPU (slower)")
    N_ENVS = 4
    BATCH_SIZE = 256

CURRICULUM = [
    {"track": "tracks/curriculum_1_straight.json", "timesteps": 500_000, "max_steps": 1500, "name": "Stage 1: Straight"},
    {"track": "tracks/curriculum_2_gentle_oval.json", "timesteps": 750_000, "max_steps": 3000, "name": "Stage 2: Gentle oval"},
    {"track": "tracks/curriculum_3_tight_oval.json", "timesteps": 750_000, "max_steps": 3000, "name": "Stage 3: Tight oval"},
    {"track": "tracks/exp_track_8.json", "timesteps": 1_000_000, "max_steps": 5000, "name": "Stage 4: Complex track"},
]

print(f"Total timesteps: {sum(s['timesteps'] for s in CURRICULUM):,}")
print(f"Device: {DEVICE}")
print(f"Parallel envs: {N_ENVS}")
print(f"Batch size: {BATCH_SIZE}")

## 3. Curriculum Training (SB3)

In [ ]:
from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.callbacks import BaseCallback
import time


class ProgressCallback(BaseCallback):
    def __init__(self, stage_name, total_timesteps, print_freq=2048):
        super().__init__()
        self.stage_name = stage_name
        self.total = total_timesteps
        self.print_freq = print_freq
        self.start_time = None

    def _on_training_start(self):
        self.start_time = time.time()

    def _on_step(self) -> bool:
        if self.n_calls % self.print_freq == 0 and len(self.model.ep_info_buffer) > 0:
            mean_r = sum(ep['r'] for ep in self.model.ep_info_buffer) / len(self.model.ep_info_buffer)
            elapsed = time.time() - self.start_time
            steps = self.num_timesteps
            sps = steps / elapsed if elapsed > 0 else 0
            pct = 100 * steps / self.total
            eta = (self.total - steps) / sps if sps > 0 else 0
            print(f"  [{self.stage_name}] {pct:5.1f}% | steps: {steps:>8,} | reward: {mean_r:8.2f} | {sps:.0f} sps | ETA: {eta/60:.1f}m")
        return True


def make_env(track_path, max_steps):
    def _init():
        return HorseRacingSingleEnv(track_path=track_path, max_steps=max_steps)
    return _init


checkpoint_dir = Path("checkpoints/baseline")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Use SubprocVecEnv for true parallel rollouts (faster than DummyVecEnv)
VecEnvClass = SubprocVecEnv if N_ENVS > 1 else DummyVecEnv
print(f"Using {VecEnvClass.__name__} with {N_ENVS} envs\n")

model = None
total_start = time.time()

for i, stage in enumerate(CURRICULUM):
    stage_num = i + 1
    print(f"{'='*60}")
    print(f"{stage['name']} — {stage['timesteps']:,} timesteps")
    print(f"{'='*60}")

    env = VecEnvClass([make_env(stage['track'], stage['max_steps']) for _ in range(N_ENVS)])

    if model is None:
        model = PPO(
            "MlpPolicy", env, verbose=0,
            n_steps=2048, batch_size=BATCH_SIZE, n_epochs=10,
            learning_rate=3e-4, gamma=0.99, gae_lambda=0.95,
            clip_range=0.2, vf_coef=0.5, ent_coef=0.01,
            device=DEVICE,
        )
    else:
        model.set_env(env)

    callback = ProgressCallback(stage['name'], stage['timesteps'])
    model.learn(total_timesteps=stage['timesteps'], callback=callback, reset_num_timesteps=False)

    save_path = checkpoint_dir / f"curriculum_stage_{stage_num}"
    model.save(str(save_path))
    print(f"  Saved → {save_path}\n")
    env.close()

total_time = time.time() - total_start
print(f"Curriculum complete in {total_time/60:.1f} minutes")

## 4. Export to ONNX

In [ ]:
import torch
import torch.nn as nn
import onnxruntime as ort
import numpy as np


class PolicyNetwork(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.features_extractor = sb3_policy.features_extractor
        self.mlp_extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net

    def forward(self, obs):
        features = self.features_extractor(obs)
        latent_pi, _ = self.mlp_extractor(features)
        return self.action_net(latent_pi)


wrapper = PolicyNetwork(model.policy)
wrapper.eval()
obs_dim = model.observation_space.shape[0]
dummy = torch.zeros(1, obs_dim, dtype=torch.float32)

onnx_path = str(checkpoint_dir / "baseline_jockey.onnx")
torch.onnx.export(
    wrapper, dummy, onnx_path,
    input_names=["obs"], output_names=["action"],
    dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
    opset_version=17, dynamo=False,
)

# Verify
sess = ort.InferenceSession(onnx_path)
test = np.zeros((1, obs_dim), dtype=np.float32)
result = sess.run(["action"], {"obs": test})
print(f"ONNX exported: {onnx_path}")
print(f"Input: (1, {obs_dim}) → Output: {result[0][0]}")

## 5. Quick Evaluation

In [ ]:
# Run a race with the trained model and check behavior
engine = HorseRacingEngine("tracks/exp_track_8.json")
sess = ort.InferenceSession(onnx_path)

for tick in range(3000):
    obs = engine.get_observations()
    arr = engine.obs_to_array(obs[0]).reshape(1, -1)
    action = sess.run(["action"], {"obs": arr})[0][0]
    
    actions = [HorseAction(float(action[0]), float(action[1]))]
    for _ in range(1, HORSE_COUNT):
        actions.append(HorseAction())
    engine.step(actions)
    
    if tick % 500 == 499:
        o = obs[0]
        print(
            f"Tick {tick:4d} | "
            f"progress: {o['track_progress']:.3f} | "
            f"vel: {o['tangential_vel']:.1f} | "
            f"stamina: {o['stamina_ratio']:.3f} | "
            f"disp: {o['displacement']:+.2f} | "
            f"normal_vel: {o['normal_vel']:+.3f}"
        )

print(f"\nFinal progress: {engine.horses[0].track_progress:.3f}")
print(f"Final stamina: {engine.horses[0].runtime.current_stamina:.1f}")

## 6. Download Model

Download the ONNX file to use in the browser simulation.

In [ ]:
# Download the ONNX model (works in Colab)
try:
    from google.colab import files
    files.download(onnx_path)
    print("Download started!")
except ImportError:
    print(f"Not in Colab. Model saved at: {onnx_path}")